# 📓 Semana 3 · Dia 4 — Window functions no Spark, UDFs (quando evitar) e pandas API

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (Spark SQL), Spark Dev Assoc |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | Exercícios de window + pandas API |

---


## 📖 Teoria — Window no Spark

Na DataFrame API, use `Window.partitionBy(...).orderBy(...)`:

```python
from pyspark.sql.window import Window
w = Window.partitionBy('pais').orderBy(col('receita').desc())
df.withColumn('rn', row_number().over(w))
```

Mesmos conceitos do SQL: PARTITION BY fatia, ORDER BY ordena dentro da fatia.


## 📖 Teoria — UDFs Python: quando (não) usar

UDF Python roda a função **linha a linha** fora do motor (serialização + Python) — 10–100x mais lento que expressões nativas.

**Regra de decisão**:
1. Existe função nativa (built-in) ou SQL? → use.
2. Precisa de lógica complexa não nativa? → tente `when/otherwise`, depois UDF.
3. Só use UDF se não houver alternativa; marque com `@udf(returnType=...)`.
4. Para pandas: use **pandas UDF (Vectorized UDF)** — roda por lote, muito mais rápido.


## 📖 Teoria — pandas API on Spark e Spark Connect

A **pandas API on Spark** (`pyspark.pandas`) permite escrever código estilo pandas rodando distribuído — excelente para quem vem do pandas.

**Spark Connect** (2024+) separa cliente do servidor: seu código Python fala com o cluster via gRPC. É o futuro da API e cai na prova **Spark Developer Associate 2026** (peso aumentado).


### 💻 Na prática — Window na prática

Ranking de produtos por país e delta mês a mês.


In [ ]:
# Window: top produtos por país
from pyspark.sql.functions import row_number, rank, lag, sum as s
from pyspark.sql.window import Window
vendas_prod = (spark.table("workspace.bronze.vendas_bronze")
    .groupBy("Country", "StockCode")
    .agg(s("Quantity").alias("qtd")))
w = Window.partitionBy("Country").orderBy(col("qtd").desc())
top = vendas_prod.withColumn("rn", row_number().over(w)).filter("rn <= 3")
top.show(9)

In [ ]:
# Running total por país
rec_mes = (spark.table("workspace.bronze.vendas_bronze")
    .withColumn("mes", to_date("InvoiceDate", "M/d/yyyy H:mm"))
    .groupBy("Country", "mes").agg(s("Quantity*UnitPrice").alias("receita")))
w2 = Window.partitionBy("Country").orderBy("mes").rowsBetween(Window.unboundedPreceding, Window.currentRow)
rec_mes.withColumn("acumulado", s("receita").over(w2)).show(8)

### 💻 Na prática — UDF vs nativa — o teste

Compare o tempo de uma expressão nativa vs uma UDF Python.


In [ ]:
# Expressão nativa (rápida)
import time
df = spark.table("workspace.bronze.vendas_bronze")
t0 = time.time()
nativo = df.withColumn("receita_linha", col("Quantity") * col("UnitPrice")).count()
t1 = time.time()
print(f"Nativo: {nativo} linhas em {t1-t0:.2f}s")

In [ ]:
# UDF Python (lenta) — para comparar
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType
import time
@udf(returnType=DoubleType())
def receita_udf(q, p):
    return q * p
t0 = time.time()
u = df.withColumn("receita_linha", receita_udf(col("Quantity"), col("UnitPrice"))).count()
t1 = time.time()
print(f"UDF: {u} linhas em {t1-t0:.2f}s (geralmente 5-50x mais lento)")

### 💻 Na prática — pandas API on Spark

Use a API pandas distribuída para operações familiares.


In [ ]:
# pandas API on Spark (exemplo)
import pyspark.pandas as ps
dfp = df[["Country", "Quantity", "UnitPrice"]]\
    .to_pandas_on_spark()
print(dfp.groupby("Country")["Quantity"].sum().sort_values(ascending=False).head(5))

> 🎯 **Dica de prova**: Spark Connect e pandas API on Spark ganharam peso na **Spark Developer Associate** (2026). No DEA, saber que UDFs são mais lentas e que vectorized UDF existe é suficiente.


## 🎯 Exercícios de fixação

**1.** Use `rank()` e `dense_rank()` e mostre a diferença com empate.

**2.** Reescreva a UDF `receita_udf` sem UDF (só com col).

**3.** Conte quantos produtos são top-1 em mais de um país (com window).


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** rank vs dense_rank

Com qtd (10,10,9): rank → 1,1,3; dense_rank → 1,1,2. Mesma lógica do SQL.

**2.** Sem UDF

`df.withColumn('receita_linha', col('Quantity') * col('UnitPrice'))` — nativa, otimizada pelo Catalyst.

**3.** Top-1 em vários países

```python
w = Window.partitionBy('Country').orderBy(col('qtd').desc())
top1 = vendas_prod.withColumn('rn', row_number().over(w)).filter('rn = 1')
top1.groupBy('StockCode').count().filter('count > 1')
```



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*